# Анализ пролонгаций договоров за 2023 год

**Цель:** рассчитать коэффициенты пролонгации по аккаунт-менеджерам и по отделу в целом, а затем сформировать Excel-отчет для руководителя.

В расчетах используются два коэффициента:

1. **Коэффициент пролонгации в первый месяц** = сумма отгрузки проектов, которые были пролонгированы в первый месяц после завершения / сумма отгрузки последнего месяца всех проектов, завершившихся в предыдущем месяце.
2. **Коэффициент пролонгации во второй месяц** = сумма отгрузки проектов, которые были пролонгированы во второй месяц / сумма отгрузки последнего месяца проектов, которые не были пролонгированы в первый месяц.

Данные об ответственном менеджере берем из `prolongations.csv`, потому что по ТЗ эти данные являются первичными относительно `financial_data.csv`.

## 1. Импорт библиотек

Используем:
- `pandas` и `numpy` для обработки данных;
- `openpyxl` для создания оформленного Excel-отчета;
- `re` для очистки денежных значений, которые в исходном CSV записаны как текст.

In [3]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.chart import LineChart, BarChart, Reference

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2. Пути к исходным файлам

Положите файлы `financial_data.csv` и `prolongations.csv` в одну папку с проектом.

In [5]:
financial_path = Path("financial_data.csv")
prolongations_path = Path("prolongations.csv")

if not financial_path.exists():
    financial_path = Path("/mnt/data/financial_data.csv")

if not prolongations_path.exists():
    prolongations_path = Path("/mnt/data/prolongations.csv")

output_path = Path("prolongation_report_2023.xlsx")

print("Файл financial_data найден:", financial_path.exists(), financial_path)
print("Файл prolongations найден:", prolongations_path.exists(), prolongations_path)

Файл financial_data найден: True financial_data.csv
Файл prolongations найден: True prolongations.csv


## 3. Загрузка данных

Смотрим размер таблиц и первые строки, чтобы убедиться, что структура загрузилась корректно.

In [6]:
financial = pd.read_csv(financial_path)
prolongations = pd.read_csv(prolongations_path)

print("financial_data:", financial.shape)
print("prolongations:", prolongations.shape)

display(financial.head())
display(prolongations.head())

financial_data: (451, 19)
prolongations: (477, 3)


,id,Причина дубля,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024,Account
0,42,NaN,"36 220,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
1,657,первая часть оплаты,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
2,657,вторая часть оплаты,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
3,594,NaN,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
4,665,NaN,"10 000,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович


,id,month,AM
0,42,ноябрь 2022,Васильев Артем Александрович
1,453,ноябрь 2022,Васильев Артем Александрович
2,548,ноябрь 2022,Михайлов Андрей Сергеевич
3,87,ноябрь 2022,Соколова Анастасия Викторовна
4,429,ноябрь 2022,Соколова Анастасия Викторовна


## 4. Список месяцев

Для расчета 2023 года нужны месяцы с ноября 2022 по декабрь 2023:

- январь 2023 считается по проектам, завершившимся в декабре 2022, и по проектам ноября 2022 для второго месяца;
- декабрь 2023 считается по проектам, завершившимся в ноябре 2023, и по проектам октября 2023 для второго месяца.

In [21]:
all_months = [
    "Ноябрь 2022", "Декабрь 2022",
    "Январь 2023", "Февраль 2023", "Март 2023", "Апрель 2023",
    "Май 2023", "Июнь 2023", "Июль 2023", "Август 2023",
    "Сентябрь 2023", "Октябрь 2023", "Ноябрь 2023", "Декабрь 2023",
    "Январь 2024", "Февраль 2024"
]

report_months = [
    "Январь 2023", "Февраль 2023", "Март 2023", "Апрель 2023",
    "Май 2023", "Июнь 2023", "Июль 2023", "Август 2023",
    "Сентябрь 2023", "Октябрь 2023", "Ноябрь 2023", "Декабрь 2023"
]

month_short_names = [m.replace(" 2023", "") for m in report_months]

missing_month_cols = [m for m in all_months if m not in financial.columns]
print("Отсутствующие месячные колонки:", missing_month_cols)

Отсутствующие месячные колонки: []


## 5. Очистка денежных значений

В исходном файле суммы записаны не как числа, а как текст, например:

- `36 220,00`
- `10 000,00`
- `стоп`
- `в ноль`

Поэтому перед расчетами привожу все месячные колонки к числовому типу. Текстовые значения без суммы считаем как `0`.

In [23]:
def clean_money_value(value):
    if pd.isna(value):
        return 0.0

    text = str(value).strip().lower()
    text = text.replace("\xa0", "")
    text = text.replace(" ", "")
    text = text.replace(",", ".")
    text = re.sub(r"[^0-9.\-]", "", text)

    if text in ["", ".", "-", "-."]:
        return 0.0

    try:
        return float(text)
    except ValueError:
        return 0.0

financial_clean = financial.copy()

for col in all_months:
    financial_clean[col] = financial_clean[col].apply(clean_money_value)

print("Типы данных после очистки месячных колонок:")
display(financial_clean[all_months].dtypes.to_frame("dtype").T)
display(financial_clean.head())

Типы данных после очистки месячных колонок:


,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024
dtype,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64


,id,Причина дубля,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024,Account
0,42,NaN,"36,220.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,Васильев Артем Александрович
1,657,первая часть оплаты,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,Васильев Артем Александрович
2,657,вторая часть оплаты,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,Васильев Артем Александрович
3,594,NaN,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,Васильев Артем Александрович
4,665,NaN,"10,000.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,Васильев Артем Александрович


## 6. Агрегация дублей по проектам

Для корректного расчета суммируем отгрузки по каждому проекту `id`. Менеджера из `financial_data` не используем в расчете, потому что по условию менеджер из `prolongations.csv` является основным источником.

In [24]:
financial_by_project = (
    financial_clean
    .groupby("id", as_index=False)[all_months]
    .sum()
)

print("Строк в financial_data до агрегации:", len(financial_clean))
print("Уникальных проектов после агрегации:", len(financial_by_project))

display(financial_by_project.head())

Строк в financial_data до агрегации: 451
Уникальных проектов после агрегации: 314


,id,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024
0,15,"439,280.00","439,280.00","102,433.75","102,433.75","102,433.75","138,158.00","138,158.00","102,433.75",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
1,16,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2,31,"55,100.00","55,100.00",0.00,"44,775.00","44,775.00","44,775.00","44,775.00","44,775.00","44,775.00","44,775.00","44,775.00","44,775.00","44,775.00","44,775.00","44,775.00","46,200.00"
3,39,"137,700.00","137,700.00","149,206.50","149,206.50","149,206.50","149,206.50","149,206.50","149,206.50","149,206.50","149,206.50","149,206.50","149,206.50","149,206.50","149,206.50",0.00,0.00
4,42,"36,220.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


## 7. Объединение таблиц

Объединяем `prolongations.csv` и агрегированные финансовые данные по `id`.

После объединения проверяем:
- есть ли проекты из `prolongations.csv`, которые не нашлись в финансовых данных;
- корректно ли распознаны месяцы завершения проектов.

In [25]:
prolongations_clean = prolongations.copy()
prolongations_clean["month"] = prolongations_clean["month"].astype(str).str.strip()
prolongations_clean["AM"] = prolongations_clean["AM"].astype(str).str.strip()

month_name_map = {m.lower(): m for m in all_months}
prolongations_clean["month_norm"] = prolongations_clean["month"].str.lower().map(month_name_map)

data = prolongations_clean.merge(
    financial_by_project,
    on="id",
    how="left",
    indicator=True
)

for col in all_months:
    data[col] = data[col].fillna(0.0)

checks = pd.DataFrame({
    "Проверка": [
        "Строк в prolongations",
        "Строк после объединения",
        "Проектов без financial_data",
        "Строк с нераспознанным месяцем",
        "Количество менеджеров"
    ],
    "Значение": [
        len(prolongations_clean),
        len(data),
        int((data["_merge"] == "left_only").sum()),
        int(data["month_norm"].isna().sum()),
        data["AM"].nunique()
    ]
})

display(checks)
display(data.head())

,Проверка,Значение
0,Строк в prolongations,477
1,Строк после объединения,477
2,Проектов без financial_data,0
3,Строк с нераспознанным месяцем,0
4,Количество менеджеров,10


,id,month,AM,month_norm,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024,_merge
0,42,ноябрь 2022,Васильев Артем Александрович,Ноябрь 2022,"36,220.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,both
1,453,ноябрь 2022,Васильев Артем Александрович,Ноябрь 2022,0.00,"39,245.00","44,320.00","177,635.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,both
2,548,ноябрь 2022,Михайлов Андрей Сергеевич,Ноябрь 2022,"674,000.00","674,000.00","674,000.00","674,000.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,both
3,87,ноябрь 2022,Соколова Анастасия Викторовна,Ноябрь 2022,"70,050.00",0.00,"73,380.00","83,480.00","89,300.00","89,300.00","78,605.00","72,485.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,both
4,429,ноябрь 2022,Соколова Анастасия Викторовна,Ноябрь 2022,"30,280.00","35,580.00","35,830.00","42,830.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,both


## 8. Логика расчета коэффициентов

Для каждого отчетного месяца 2023 года считаем две группы показателей.

### Коэффициент 1, пролонгация в первый месяц
Например, для мая 2023:

- база = сумма отгрузки за апрель по проектам, у которых последний месяц реализации — апрель;
- пролонгировано = сумма отгрузки за май по этим же проектам, если в мае есть отгрузка;
- коэффициент = пролонгировано / база.

### Коэффициент 2, пролонгация во второй месяц
Например, для мая 2023:

- берем проекты, завершившиеся в марте;
- оставляем только те, у которых не было отгрузки в апреле;
- база = сумма отгрузки этих проектов за март;
- пролонгировано = сумма отгрузки за май по этим проектам, если в мае есть отгрузка;
- коэффициент = пролонгировано / база.

In [26]:
def safe_divide(numerator, denominator):
    if denominator == 0:
        return np.nan
    return numerator / denominator


def calculate_metrics_for_month(df_part, report_month):
    month_index = all_months.index(report_month)
    previous_month = all_months[month_index - 1]
    two_months_before = all_months[month_index - 2]

    first_month_projects = df_part[df_part["month_norm"] == previous_month].copy()
    base_1 = first_month_projects[previous_month].sum()
    prolong_1 = first_month_projects.loc[first_month_projects[report_month] > 0, report_month].sum()
    coef_1 = safe_divide(prolong_1, base_1)

    second_month_projects = df_part[df_part["month_norm"] == two_months_before].copy()
    second_month_projects = second_month_projects[second_month_projects[previous_month] == 0].copy()
    base_2 = second_month_projects[two_months_before].sum()
    prolong_2 = second_month_projects.loc[second_month_projects[report_month] > 0, report_month].sum()
    coef_2 = safe_divide(prolong_2, base_2)

    return {
        "base_1_month": base_1,
        "prolong_1_month": prolong_1,
        "coef_1_month": coef_1,
        "projects_base_1_month": first_month_projects["id"].nunique(),
        "projects_prolong_1_month": first_month_projects.loc[first_month_projects[report_month] > 0, "id"].nunique(),
        "base_2_month": base_2,
        "prolong_2_month": prolong_2,
        "coef_2_month": coef_2,
        "projects_base_2_month": second_month_projects["id"].nunique(),
        "projects_prolong_2_month": second_month_projects.loc[second_month_projects[report_month] > 0, "id"].nunique(),
    }

## 9. Расчет по менеджерам и по отделу

Сначала считаем показатели для каждого менеджера по каждому месяцу. Затем отдельно считаем показатели для всего отдела.

Важно: показатель по отделу считается не как среднее по менеджерам, а как отношение общей суммы пролонгированных отгрузок к общей базе. Это правильнее для управленческого отчета.

In [27]:
rows = []

for manager, manager_data in data.groupby("AM", sort=True):
    for report_month in report_months:
        metrics = calculate_metrics_for_month(manager_data, report_month)
        rows.append({"AM": manager, "month": report_month, **metrics})

for report_month in report_months:
    metrics = calculate_metrics_for_month(data, report_month)
    rows.append({"AM": "Весь отдел", "month": report_month, **metrics})

monthly_results = pd.DataFrame(rows)

month_order = {month: number for number, month in enumerate(report_months, start=1)}
monthly_results["month_order"] = monthly_results["month"].map(month_order)
monthly_results = monthly_results.sort_values(["AM", "month_order"]).reset_index(drop=True)

print("Строк в детализации:", len(monthly_results))
display(monthly_results.head(15))

Строк в детализации: 132


,AM,month,base_1_month,prolong_1_month,coef_1_month,projects_base_1_month,projects_prolong_1_month,base_2_month,prolong_2_month,coef_2_month,projects_base_2_month,projects_prolong_2_month,month_order
0,Васильев Артем Александрович,Январь 2023,"1,693,097.68","941,761.60",0.56,22,6,"260,862.00",0.00,0.00,8,0,1
1,Васильев Артем Александрович,Февраль 2023,"756,225.00","874,015.00",1.16,5,2,"891,335.00","166,275.00",0.19,16,2,2
2,Васильев Артем Александрович,Март 2023,"991,467.42","621,897.55",0.63,9,5,"95,750.00","67,774.08",0.71,3,1,3
3,Васильев Артем Александрович,Апрель 2023,"1,601,033.35","562,360.00",0.35,11,5,"364,780.00","40,200.00",0.11,4,1,4
4,Васильев Артем Александрович,Май 2023,"2,277,723.50","1,131,823.00",0.50,11,6,"1,004,848.80",0.00,0.00,6,0,5
5,Васильев Артем Александрович,Июнь 2023,"390,286.00","151,933.28",0.39,5,2,"678,208.50","38,632.50",0.06,5,1,6
6,Васильев Артем Александрович,Июль 2023,"891,755.60","521,691.00",0.59,10,5,"230,686.00",0.00,0.00,3,0,7
7,Васильев Артем Александрович,Август 2023,"649,000.00","309,900.00",0.48,5,3,"492,017.60",0.00,0.00,5,0,8
8,Васильев Артем Александрович,Сентябрь 2023,"1,361,540.74","234,774.00",0.17,8,3,"274,500.00",0.00,0.00,2,0,9
9,Васильев Артем Александрович,Октябрь 2023,"401,370.00","355,915.00",0.89,6,1,"1,135,125.00","82,020.00",0.07,5,1,10


## 10. Итоги за год

Годовой коэффициент считаем через суммы за весь год:

- годовой коэффициент 1 = сумма `prolong_1_month` за 12 месяцев / сумма `base_1_month` за 12 месяцев;
- годовой коэффициент 2 = сумма `prolong_2_month` за 12 месяцев / сумма `base_2_month` за 12 месяцев.



In [28]:
def build_year_summary(source):
    grouped = (
        source
        .groupby("AM", as_index=False)
        .agg(
            base_1_month=("base_1_month", "sum"),
            prolong_1_month=("prolong_1_month", "sum"),
            projects_base_1_month=("projects_base_1_month", "sum"),
            projects_prolong_1_month=("projects_prolong_1_month", "sum"),
            base_2_month=("base_2_month", "sum"),
            prolong_2_month=("prolong_2_month", "sum"),
            projects_base_2_month=("projects_base_2_month", "sum"),
            projects_prolong_2_month=("projects_prolong_2_month", "sum"),
        )
    )
    grouped["coef_1_month"] = grouped.apply(lambda x: safe_divide(x["prolong_1_month"], x["base_1_month"]), axis=1)
    grouped["coef_2_month"] = grouped.apply(lambda x: safe_divide(x["prolong_2_month"], x["base_2_month"]), axis=1)
    return grouped

year_summary = build_year_summary(monthly_results)

manager_rows = sorted([x for x in year_summary["AM"].unique() if x != "Весь отдел"])
year_summary["sort_key"] = year_summary["AM"].apply(lambda x: "яяя" if x == "Весь отдел" else x)
year_summary = year_summary.sort_values("sort_key").drop(columns="sort_key").reset_index(drop=True)

display(year_summary)

,AM,base_1_month,prolong_1_month,projects_base_1_month,projects_prolong_1_month,base_2_month,prolong_2_month,projects_base_2_month,projects_prolong_2_month,coef_1_month,coef_2_month
0,Васильев Артем Александрович,"11,761,147.79","6,119,964.43",107,42,"5,818,845.40","482,441.58",67,7,0.52,0.08
1,Иванова Мария Сергеевна,"4,727,657.16","1,660,095.55",47,15,"2,503,864.75",0.00,33,0,0.35,0.00
2,Кузнецов Михаил Иванович,"982,724.44","470,182.98",15,7,"167,675.00",0.00,4,0,0.48,0.00
3,Михайлов Андрей Сергеевич,"3,361,833.59","2,235,422.29",26,14,"1,056,004.68",0.00,13,0,0.66,0.00
4,Петрова Анна Дмитриевна,"98,492.00","109,442.52",1,1,0.00,0.00,0,0,1.11,NaN
5,Попова Екатерина Николаевна,"3,680,853.08","1,885,685.80",61,25,"1,750,920.26","241,399.00",34,4,0.51,0.14
6,Смирнова Ольга Владимировна,"2,951,204.59","2,124,958.50",51,22,"803,500.80","203,825.00",21,4,0.72,0.25
7,Соколова Анастасия Викторовна,"6,798,714.58","3,974,267.97",72,34,"2,812,661.98","169,725.00",36,3,0.58,0.06
8,Федорова Марина Васильевна,0.00,0.00,0,0,0.00,0.00,0,0,NaN,NaN
9,без А/М,0.00,0.00,1,0,0.00,0.00,1,0,NaN,NaN


## 11. Проверка результата

Здесь проверяем, что после очистки и расчетов базы не равны нулю по всему отчету. Если в этой таблице есть нормальные суммы и коэффициенты, значит расчет работает корректно.

In [30]:
department_monthly = monthly_results[monthly_results["AM"] == "Весь отдел"].copy()

check_columns = [
    "month",
    "base_1_month", "prolong_1_month", "coef_1_month",
    "base_2_month", "prolong_2_month", "coef_2_month"
]

display(department_monthly[check_columns])

,month,base_1_month,prolong_1_month,coef_1_month,base_2_month,prolong_2_month,coef_2_month
12,Январь 2023,"5,986,766.86","2,677,021.61",0.45,"545,022.00","73,380.00",0.13
13,Февраль 2023,"2,549,075.50","1,880,945.00",0.74,"2,890,042.76","262,620.00",0.09
14,Март 2023,"1,899,332.77","1,271,739.40",0.67,"760,840.50","122,999.08",0.16
15,Апрель 2023,"3,377,742.20","1,501,923.35",0.44,"627,735.00","57,940.00",0.09
16,Май 2023,"3,647,436.50","2,194,460.75",0.60,"1,796,703.80",0.00,0.00
17,Июнь 2023,"1,274,625.53","333,068.28",0.26,"1,015,068.50","38,632.50",0.04
18,Июль 2023,"2,625,982.85","1,390,411.00",0.53,"1,020,325.53","108,285.00",0.11
19,Август 2023,"1,936,915.83","943,678.88",0.49,"1,452,499.85","52,315.00",0.04
20,Сентябрь 2023,"3,241,978.50","1,086,504.42",0.34,"755,739.63",0.00,0.00
21,Октябрь 2023,"3,820,835.81","3,162,695.19",0.83,"2,183,145.12","245,379.00",0.11


## 12. Подготовка таблиц для Excel-отчета

Формируем три основные таблицы по шаблону:

1. **Весь отдел** - строки по месяцам, показатели по двум коэффициентам.
2. **Менеджеры за год** - строки по менеджерам, итоги за 2023 год.
3. **Менеджеры по месяцам** - две матрицы: коэффициент 1 и коэффициент 2 по менеджерам и месяцам.

Дополнительно создаем листы с детализацией и проверками данных.

In [31]:
coef_1_pivot = (
    monthly_results
    .pivot(index="AM", columns="month", values="coef_1_month")
    .reindex(columns=report_months)
)

coef_2_pivot = (
    monthly_results
    .pivot(index="AM", columns="month", values="coef_2_month")
    .reindex(columns=report_months)
)

row_order = sorted([x for x in coef_1_pivot.index if x != "Весь отдел"]) + ["Весь отдел"]
coef_1_pivot = coef_1_pivot.reindex(row_order)
coef_2_pivot = coef_2_pivot.reindex(row_order)

year_coef_map_1 = year_summary.set_index("AM")["coef_1_month"].to_dict()
year_coef_map_2 = year_summary.set_index("AM")["coef_2_month"].to_dict()
coef_1_pivot["За год"] = coef_1_pivot.index.map(year_coef_map_1)
coef_2_pivot["За год"] = coef_2_pivot.index.map(year_coef_map_2)

display(coef_1_pivot.head())
display(coef_2_pivot.head())

month,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,За год
AM,,,,,,,,,,,,,
Васильев Артем Александрович,0.56,1.16,0.63,0.35,0.50,0.39,0.59,0.48,0.17,0.89,0.62,0.32,0.52
Иванова Мария Сергеевна,0.27,NaN,0.45,0.28,1.00,0.00,0.47,0.54,1.00,NaN,NaN,NaN,0.35
Кузнецов Михаил Иванович,1.27,0.85,NaN,0.00,NaN,NaN,NaN,NaN,NaN,1.31,0.47,0.27,0.48
Михайлов Андрей Сергеевич,0.69,0.90,2.43,1.10,1.19,0.00,1.20,NaN,0.00,0.45,NaN,0.00,0.66
Петрова Анна Дмитриевна,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.11,1.11


month,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,За год
AM,,,,,,,,,,,,,
Васильев Артем Александрович,0.00,0.19,0.71,0.11,0.00,0.06,0.00,0.00,0.00,0.07,0.00,0.48,0.08
Иванова Мария Сергеевна,0.00,0.00,NaN,0.00,0.00,NaN,0.00,0.00,0.00,NaN,NaN,NaN,0.00
Кузнецов Михаил Иванович,NaN,NaN,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00
Михайлов Андрей Сергеевич,0.00,0.00,NaN,NaN,NaN,NaN,0.00,NaN,NaN,0.00,0.00,NaN,0.00
Петрова Анна Дмитриевна,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 13. Функции для оформления Excel

Ниже задаем общий стиль отчета: заливка заголовков, границы, форматы чисел и процентов.

In [32]:
header_fill = PatternFill("solid", fgColor="FFF2CC")
section_fill = PatternFill("solid", fgColor="FFD966")
light_fill = PatternFill("solid", fgColor="F8F9FA")
white_fill = PatternFill("solid", fgColor="FFFFFF")
blue_fill = PatternFill("solid", fgColor="D9EAF7")

thin_gray = Side(style="thin", color="D9D9D9")
medium_gray = Side(style="medium", color="A6A6A6")

thin_border = Border(left=thin_gray, right=thin_gray, top=thin_gray, bottom=thin_gray)
section_border = Border(left=thin_gray, right=thin_gray, top=medium_gray, bottom=medium_gray)

font_default = "Calibri"


def style_cell(cell, fill=None, bold=False, font_size=11, align="center", wrap=True, border=True):
    cell.font = Font(name=font_default, size=font_size, bold=bold, color="000000")
    if fill is not None:
        cell.fill = fill
    cell.alignment = Alignment(horizontal=align, vertical="center", wrap_text=wrap)
    if border:
        cell.border = thin_border


def set_sheet_defaults(ws):
    ws.freeze_panes = "A5"
    ws.sheet_view.showGridLines = True
    for row in ws.iter_rows():
        for cell in row:
            cell.font = Font(name=font_default, size=11)


def format_money(cell):
    cell.number_format = '#,##0.00'


def format_percent(cell):
    cell.number_format = '0.0%'


def add_color_scale(ws, cell_range):
    ws.conditional_formatting.add(
        cell_range,
        ColorScaleRule(
            start_type="num", start_value=0, start_color="F4CCCC",
            mid_type="percentile", mid_value=50, mid_color="FFF2CC",
            end_type="max", end_color="D9EAD3"
        )
    )


def autosize_columns(ws, min_width=10, max_width=38):
    for column_cells in ws.columns:
        letter = get_column_letter(column_cells[0].column)
        max_length = 0
        for cell in column_cells:
            value = cell.value
            if value is not None:
                max_length = max(max_length, len(str(value)))
        width = max(min_width, min(max_width, max_length + 2))
        ws.column_dimensions[letter].width = width


## 14. Создание Excel-отчета

Excel создается в формате:

- лист `Весь отдел`;
- лист `Менеджеры за год`;
- лист `Менеджеры по месяцам`;
- дополнительные листы `Дашборд`, `Детализация`, `Проверки данных`.

In [33]:
wb = Workbook()
wb.remove(wb.active)

ws_dept = wb.create_sheet("Весь отдел")
ws_year = wb.create_sheet("Менеджеры за год")
ws_months = wb.create_sheet("Менеджеры по месяцам")
ws_dash = wb.create_sheet("Дашборд")
ws_detail = wb.create_sheet("Детализация")
ws_checks = wb.create_sheet("Проверки данных")

for ws in wb.worksheets:
    set_sheet_defaults(ws)


# Лист: Весь отдел
ws = ws_dept
ws.merge_cells("A1:G1")
ws["A1"] = "Отчет по пролонгациям за 2023 год — весь отдел"
style_cell(ws["A1"], fill=section_fill, bold=True, font_size=14, align="left", border=False)

ws.merge_cells("A3:A4")
ws.merge_cells("B3:D3")
ws.merge_cells("E3:G3")
ws["A3"] = "Месяц"
ws["B3"] = "Пролонгации в первый месяц"
ws["E3"] = "Пролонгации через месяц"
headers = ["к пролонгации", "пролонгировано", "Коэффициент", "к пролонгации", "пролонгировано", "Коэффициент"]
for col, header in enumerate(headers, start=2):
    ws.cell(row=4, column=col, value=header)

for row in ws.iter_rows(min_row=3, max_row=4, min_col=1, max_col=7):
    for cell in row:
        style_cell(cell, fill=header_fill, bold=True)

report_data = department_monthly.copy()
report_data = report_data.sort_values("month_order")

start_row = 5
for r, (_, row) in enumerate(report_data.iterrows(), start=start_row):
    values = [
        row["month"].replace(" 2023", ""),
        row["base_1_month"], row["prolong_1_month"], row["coef_1_month"],
        row["base_2_month"], row["prolong_2_month"], row["coef_2_month"]
    ]
    for c, value in enumerate(values, start=1):
        cell = ws.cell(row=r, column=c, value=value)
        style_cell(cell, align="left" if c == 1 else "center", fill=white_fill)
        if c in [2, 3, 5, 6]:
            format_money(cell)
        if c in [4, 7]:
            format_percent(cell)

summary_row = start_row + len(report_data)
dept_year = year_summary[year_summary["AM"] == "Весь отдел"].iloc[0]
summary_values = [
    "За год",
    dept_year["base_1_month"], dept_year["prolong_1_month"], dept_year["coef_1_month"],
    dept_year["base_2_month"], dept_year["prolong_2_month"], dept_year["coef_2_month"]
]
for c, value in enumerate(summary_values, start=1):
    cell = ws.cell(row=summary_row, column=c, value=value)
    style_cell(cell, fill=blue_fill, bold=True, align="left" if c == 1 else "center")
    if c in [2, 3, 5, 6]:
        format_money(cell)
    if c in [4, 7]:
        format_percent(cell)

add_color_scale(ws, f"D{start_row}:D{summary_row}")
add_color_scale(ws, f"G{start_row}:G{summary_row}")
ws.column_dimensions["A"].width = 16
for col in range(2, 8):
    ws.column_dimensions[get_column_letter(col)].width = 17


# Лист: Менеджеры за год
ws = ws_year
ws.merge_cells("A1:G1")
ws["A1"] = "Отчет по пролонгациям за 2023 год — менеджеры за год"
style_cell(ws["A1"], fill=section_fill, bold=True, font_size=14, align="left", border=False)

ws.merge_cells("A3:A4")
ws.merge_cells("B3:D3")
ws.merge_cells("E3:G3")
ws["A3"] = "Менеджер"
ws["B3"] = "Пролонгации в первый месяц"
ws["E3"] = "Пролонгации через месяц"
for col, header in enumerate(headers, start=2):
    ws.cell(row=4, column=col, value=header)

for row in ws.iter_rows(min_row=3, max_row=4, min_col=1, max_col=7):
    for cell in row:
        style_cell(cell, fill=header_fill, bold=True)

start_row = 5
year_table = year_summary.copy()
for r, (_, row) in enumerate(year_table.iterrows(), start=start_row):
    values = [
        row["AM"],
        row["base_1_month"], row["prolong_1_month"], row["coef_1_month"],
        row["base_2_month"], row["prolong_2_month"], row["coef_2_month"]
    ]
    is_dept = row["AM"] == "Весь отдел"
    for c, value in enumerate(values, start=1):
        cell = ws.cell(row=r, column=c, value=value)
        style_cell(cell, fill=blue_fill if is_dept else white_fill, bold=is_dept, align="left" if c == 1 else "center")
        if c in [2, 3, 5, 6]:
            format_money(cell)
        if c in [4, 7]:
            format_percent(cell)

last_year_row = start_row + len(year_table) - 1
add_color_scale(ws, f"D{start_row}:D{last_year_row}")
add_color_scale(ws, f"G{start_row}:G{last_year_row}")
ws.column_dimensions["A"].width = 34
for col in range(2, 8):
    ws.column_dimensions[get_column_letter(col)].width = 17


# Лист: Менеджеры по месяцам
ws = ws_months
ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=14)
ws["A1"] = "Коэффициент 1"
style_cell(ws["A1"], fill=section_fill, bold=True, font_size=12, align="left")

ws.merge_cells(start_row=2, start_column=1, end_row=3, end_column=1)
ws.merge_cells(start_row=2, start_column=2, end_row=2, end_column=14)
ws["A2"] = "Менеджер"
ws["B2"] = "Месяц"
for c, month in enumerate(month_short_names + ["За год"], start=2):
    ws.cell(row=3, column=c, value=month)
for row in ws.iter_rows(min_row=2, max_row=3, min_col=1, max_col=14):
    for cell in row:
        style_cell(cell, fill=header_fill, bold=True)

start_row_coef_1 = 4
for r, manager in enumerate(coef_1_pivot.index, start=start_row_coef_1):
    ws.cell(row=r, column=1, value=manager)
    style_cell(ws.cell(row=r, column=1), align="left", fill=blue_fill if manager == "Весь отдел" else white_fill, bold=(manager == "Весь отдел"))
    for c, col in enumerate(report_months + ["За год"], start=2):
        value = coef_1_pivot.loc[manager, col]
        cell = ws.cell(row=r, column=c, value=value)
        style_cell(cell, fill=blue_fill if manager == "Весь отдел" else white_fill, bold=(manager == "Весь отдел"))
        format_percent(cell)

last_coef_1_row = start_row_coef_1 + len(coef_1_pivot.index) - 1
add_color_scale(ws, f"B{start_row_coef_1}:N{last_coef_1_row}")

section_2_start = last_coef_1_row + 3
ws.merge_cells(start_row=section_2_start, start_column=1, end_row=section_2_start, end_column=14)
ws.cell(row=section_2_start, column=1, value="Коэффициент 2")
style_cell(ws.cell(row=section_2_start, column=1), fill=section_fill, bold=True, font_size=12, align="left")

ws.merge_cells(start_row=section_2_start + 1, start_column=1, end_row=section_2_start + 2, end_column=1)
ws.merge_cells(start_row=section_2_start + 1, start_column=2, end_row=section_2_start + 1, end_column=14)
ws.cell(row=section_2_start + 1, column=1, value="Менеджер")
ws.cell(row=section_2_start + 1, column=2, value="Месяц")
for c, month in enumerate(month_short_names + ["За год"], start=2):
    ws.cell(row=section_2_start + 2, column=c, value=month)
for row in ws.iter_rows(min_row=section_2_start + 1, max_row=section_2_start + 2, min_col=1, max_col=14):
    for cell in row:
        style_cell(cell, fill=header_fill, bold=True)

start_row_coef_2 = section_2_start + 3
for r, manager in enumerate(coef_2_pivot.index, start=start_row_coef_2):
    ws.cell(row=r, column=1, value=manager)
    style_cell(ws.cell(row=r, column=1), align="left", fill=blue_fill if manager == "Весь отдел" else white_fill, bold=(manager == "Весь отдел"))
    for c, col in enumerate(report_months + ["За год"], start=2):
        value = coef_2_pivot.loc[manager, col]
        cell = ws.cell(row=r, column=c, value=value)
        style_cell(cell, fill=blue_fill if manager == "Весь отдел" else white_fill, bold=(manager == "Весь отдел"))
        format_percent(cell)

last_coef_2_row = start_row_coef_2 + len(coef_2_pivot.index) - 1
add_color_scale(ws, f"B{start_row_coef_2}:N{last_coef_2_row}")
ws.column_dimensions["A"].width = 34
for col in range(2, 15):
    ws.column_dimensions[get_column_letter(col)].width = 12


# Лист: Дашборд
ws = ws_dash
ws.merge_cells("A1:H1")
ws["A1"] = "Дашборд пролонгаций за 2023 год"
style_cell(ws["A1"], fill=section_fill, bold=True, font_size=14, align="left", border=False)

kpis = [
    ("Коэф. 1 за год", dept_year["coef_1_month"]),
    ("Коэф. 2 за год", dept_year["coef_2_month"]),
    ("База 1 за год", dept_year["base_1_month"]),
    ("База 2 за год", dept_year["base_2_month"]),
]
for idx, (name, value) in enumerate(kpis, start=1):
    col = 1 + (idx - 1) * 2
    ws.cell(row=3, column=col, value=name)
    ws.cell(row=4, column=col, value=value)
    ws.merge_cells(start_row=3, start_column=col, end_row=3, end_column=col + 1)
    ws.merge_cells(start_row=4, start_column=col, end_row=4, end_column=col + 1)
    style_cell(ws.cell(row=3, column=col), fill=header_fill, bold=True, border=True)
    style_cell(ws.cell(row=4, column=col), fill=white_fill, bold=True, font_size=14, border=True)
    if "Коэф" in name:
        format_percent(ws.cell(row=4, column=col))
    else:
        format_money(ws.cell(row=4, column=col))

chart_table_start = 7
chart_headers = ["Месяц", "Коэф. 1", "Коэф. 2"]
for c, header in enumerate(chart_headers, start=1):
    cell = ws.cell(row=chart_table_start, column=c, value=header)
    style_cell(cell, fill=header_fill, bold=True)

for r, (_, row) in enumerate(report_data.iterrows(), start=chart_table_start + 1):
    ws.cell(row=r, column=1, value=row["month"].replace(" 2023", ""))
    ws.cell(row=r, column=2, value=row["coef_1_month"])
    ws.cell(row=r, column=3, value=row["coef_2_month"])
    for c in range(1, 4):
        style_cell(ws.cell(row=r, column=c), fill=white_fill, align="left" if c == 1 else "center")
        if c in [2, 3]:
            format_percent(ws.cell(row=r, column=c))

line_chart = LineChart()
line_chart.title = "Динамика коэффициентов по отделу"
line_chart.y_axis.title = "Коэффициент"
line_chart.x_axis.title = "Месяц"
line_chart.style = 13
line_chart.height = 9
line_chart.width = 18
values = Reference(ws, min_col=2, max_col=3, min_row=chart_table_start, max_row=chart_table_start + len(report_data))
cats = Reference(ws, min_col=1, min_row=chart_table_start + 1, max_row=chart_table_start + len(report_data))
line_chart.add_data(values, titles_from_data=True)
line_chart.set_categories(cats)
ws.add_chart(line_chart, "E7")


# Лист: Детализация
ws = ws_detail
detail_export = monthly_results.drop(columns=["month_order"]).copy()
detail_export = detail_export[[
    "AM", "month",
    "base_1_month", "prolong_1_month", "coef_1_month", "projects_base_1_month", "projects_prolong_1_month",
    "base_2_month", "prolong_2_month", "coef_2_month", "projects_base_2_month", "projects_prolong_2_month"
]]
detail_headers = [
    "Менеджер", "Месяц",
    "База 1 месяц", "Пролонгировано 1 месяц", "Коэф. 1 месяц", "Проектов к пролонгации 1", "Проектов пролонгировано 1",
    "База 2 месяц", "Пролонгировано 2 месяц", "Коэф. 2 месяц", "Проектов к пролонгации 2", "Проектов пролонгировано 2"
]
for c, header in enumerate(detail_headers, start=1):
    ws.cell(row=1, column=c, value=header)
    style_cell(ws.cell(row=1, column=c), fill=header_fill, bold=True)

for r, (_, row) in enumerate(detail_export.iterrows(), start=2):
    for c, value in enumerate(row, start=1):
        cell = ws.cell(row=r, column=c, value=value)
        style_cell(cell, fill=white_fill, align="left" if c in [1, 2] else "center")
        if c in [3, 4, 8, 9]:
            format_money(cell)
        if c in [5, 10]:
            format_percent(cell)

ws.freeze_panes = "A2"
autosize_columns(ws, min_width=12, max_width=32)


# Лист: Проверки данных
ws = ws_checks
ws.merge_cells("A1:B1")
ws["A1"] = "Проверки качества данных"
style_cell(ws["A1"], fill=section_fill, bold=True, font_size=14, align="left", border=False)

checks_for_excel = checks.copy()
extra_checks = pd.DataFrame({
    "Проверка": [
        "Строк в financial_data до агрегации",
        "Уникальных проектов в financial_data после агрегации",
        "Сумма базы 1 за год по отделу",
        "Сумма базы 2 за год по отделу"
    ],
    "Значение": [
        len(financial_clean),
        len(financial_by_project),
        dept_year["base_1_month"],
        dept_year["base_2_month"]
    ]
})
checks_for_excel = pd.concat([checks_for_excel, extra_checks], ignore_index=True)

for c, header in enumerate(checks_for_excel.columns, start=1):
    ws.cell(row=3, column=c, value=header)
    style_cell(ws.cell(row=3, column=c), fill=header_fill, bold=True)

for r, (_, row) in enumerate(checks_for_excel.iterrows(), start=4):
    ws.cell(row=r, column=1, value=row["Проверка"])
    ws.cell(row=r, column=2, value=row["Значение"])
    style_cell(ws.cell(row=r, column=1), fill=white_fill, align="left")
    style_cell(ws.cell(row=r, column=2), fill=white_fill, align="center")

ws.column_dimensions["A"].width = 48
ws.column_dimensions["B"].width = 22

for ws in wb.worksheets:
    if ws.title not in ["Весь отдел", "Менеджеры за год", "Менеджеры по месяцам", "Проверки данных"]:
        autosize_columns(ws)
    for row in ws.iter_rows():
        for cell in row:
            cell.alignment = Alignment(
                horizontal=cell.alignment.horizontal or "center",
                vertical="center",
                wrap_text=True
            )

wb.save(output_path)
print(f"Excel-отчет создан: {output_path.resolve()}")

Excel-отчет создан: /content/prolongation_report_2023.xlsx


## 15. Финальная проверка

Проверяем, что файл создан, а ключевые таблицы не пустые.

In [34]:
print("Файл создан:", output_path.exists())
print("Путь к файлу:", output_path.resolve())
print("Строк в детализации:", len(monthly_results))
print("Менеджеров в отчете:", data["AM"].nunique())

display(year_summary.head())

Файл создан: True
Путь к файлу: /content/prolongation_report_2023.xlsx
Строк в детализации: 132
Менеджеров в отчете: 10


,AM,base_1_month,prolong_1_month,projects_base_1_month,projects_prolong_1_month,base_2_month,prolong_2_month,projects_base_2_month,projects_prolong_2_month,coef_1_month,coef_2_month
0,Васильев Артем Александрович,"11,761,147.79","6,119,964.43",107,42,"5,818,845.40","482,441.58",67,7,0.52,0.08
1,Иванова Мария Сергеевна,"4,727,657.16","1,660,095.55",47,15,"2,503,864.75",0.00,33,0,0.35,0.00
2,Кузнецов Михаил Иванович,"982,724.44","470,182.98",15,7,"167,675.00",0.00,4,0,0.48,0.00
3,Михайлов Андрей Сергеевич,"3,361,833.59","2,235,422.29",26,14,"1,056,004.68",0.00,13,0,0.66,0.00
4,Петрова Анна Дмитриевна,"98,492.00","109,442.52",1,1,0.00,0.00,0,0,1.11,NaN
